# 01 — FAISS baseline từ CLIPFeatures BTC phát sẵn

**Phụ trách:** Tuấn Anh (N1) · **Mốc:** M1 · **GPU:** KHÔNG cần (CPU đủ)

Đầu vào: thư mục CLIPFeatures của BTC trên Drive chung.
Đầu ra: `faiss_clip.index` + `keyframe_ids.npy` lưu về Drive.

> Mở notebook này từ GitHub: Colab → File → Open notebook → GitHub → dán URL repo.

In [ ]:
# === SETUP: mount Drive + cài thư viện ===
from google.colab import drive
drive.mount('/content/drive')
%pip -q install faiss-cpu numpy pandas

# === CONFIG: sửa đường dẫn theo Drive chung của đội ===
DRIVE_ROOT = '/content/drive/MyDrive/AIC2026'   # thư mục chia sẻ chung
CLIP_DIR   = f'{DRIVE_ROOT}/data/clip_features'  # nơi để CLIPFeatures BTC phát
OUT_DIR    = f'{DRIVE_ROOT}/outputs/index_v1_clip'
import os; os.makedirs(OUT_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
# === BƯỚC 1: đọc toàn bộ CLIPFeatures ===
# TODO: chỉnh theo format thật của BTC. Thường là 1 file .npy per video,
# tên file = video_id, mỗi dòng = 1 keyframe (thứ tự khớp danh sách keyframe).
import numpy as np, glob, os

feature_files = sorted(glob.glob(f'{CLIP_DIR}/*.npy'))
print(f'{len(feature_files)} file features')
assert feature_files, 'Không thấy file nào — kiểm tra CLIP_DIR'

all_vecs, all_ids = [], []
for fp in feature_files:
    video_id = os.path.splitext(os.path.basename(fp))[0]
    vecs = np.load(fp).astype('float32')
    all_vecs.append(vecs)
    all_ids += [f'{video_id}#{i:06d}' for i in range(len(vecs))]

X = np.concatenate(all_vecs)
print('Tổng keyframe:', X.shape, '— chiều vector:', X.shape[1])
# Chiều vector tiết lộ model: 512=ViT-B/32 hoặc B/16, 768=ViT-L/14 → báo cả đội!

In [ ]:
# === BƯỚC 2: chuẩn hóa L2 + build FAISS IndexFlatIP ===
import faiss
faiss.normalize_L2(X)
index = faiss.IndexFlatIP(X.shape[1])
index.add(X)
print('Index size:', index.ntotal)

faiss.write_index(index, f'{OUT_DIR}/faiss_clip.index')
np.save(f'{OUT_DIR}/keyframe_ids.npy', np.array(all_ids))
print('Đã lưu về', OUT_DIR)

In [ ]:
# === BƯỚC 3: smoke test — encode 1 câu text và tìm thử ===
# QUAN TRỌNG: model text PHẢI khớp model BTC dùng tạo CLIPFeatures.
# Thử lần lượt các ứng viên, model nào cho kết quả hợp lý là đúng.
%pip -q install open_clip_torch torch
import open_clip, torch

MODEL_NAME, PRETRAINED = 'ViT-B-32', 'openai'   # TODO: đổi nếu chiều vector không khớp
model, _, _ = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)

q = 'a person eating ice cream near the sea'
with torch.no_grad():
    tv = model.encode_text(tokenizer([q])).float().numpy()
faiss.normalize_L2(tv)
D, I = index.search(tv, 10)
for d, i in zip(D[0], I[0]):
    print(f'{d:.3f}  {all_ids[i]}')
# Mở vài keyframe tương ứng xem có đúng ngữ nghĩa không → nếu lung tung, đổi MODEL_NAME.

In [ ]:
import os
data = '/content/drive/MyDrive/AIC2026/data'

# 1. Làm phẳng các thư mục lồng trùng tên (clip_features/clip-features -> clip_features)
for outer in ['clip_features', 'keyframes', 'metadata', 'objects']:
    op = f'{data}/{outer}'
    if not os.path.isdir(op):
        continue
    items = os.listdir(op)
    if len(items) == 1 and os.path.isdir(f'{op}/{items[0]}'):
        inner = f'{op}/{items[0]}'
        tmp = f'{data}/__tmp_{outer}'
        os.rename(inner, tmp)   # nhấc ruột ra ngoài
        os.rmdir(op)            # bỏ vỏ rỗng
        os.rename(tmp, op)      # đặt lại đúng tên chuẩn
        print('✓ đã làm phẳng:', outer)

# 2. Chuẩn hóa tên map-keyframes (gạch ngang theo quy ước của đội)
if os.path.isdir(f'{data}/map_keyframes') and not os.path.isdir(f'{data}/map-keyframes'):
    os.rename(f'{data}/map_keyframes', f'{data}/map-keyframes')
    print('✓ đổi tên map_keyframes -> map-keyframes')

# 3. Kiểm tra lại
print()
for sub in sorted(os.listdir(data)):
    p = f'{data}/{sub}'
    if os.path.isdir(p):
        files = os.listdir(p)
        print(f'{sub:20s} {len(files):7,} mục — ví dụ: {files[:3]}')